In [ ]:
Path = '/mnt/18T/output_2D/inf/300_1024/'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct
from sklearn.cluster import DBSCAN
def load2dxpos(fn):
    fid = open(fn, 'rb')
    p1 = np.fromfile(fid, dtype=np.float32)
    fid.close()
    n = round(len(p1)/2)
    if (n*2 != len(p1)):
        print('shape no mach')
        print(n,n**2,len(p1))
        return 0,0
    a = np.reshape(p1, (2, n),order='F')
    # print(fn,n)
    return a.T,int(np.sqrt(n))

def read_halo_file(filename):
    halos = []
    with open(filename, 'rb') as f:
        # 1. 快速读取文件头 (12 bytes)
        header_data = f.read(12)
        if len(header_data) < 12:
            return [], None, None, 0
        b_link, np_halo_min_file, np_head = struct.unpack('ffi', header_data)
        
        for _ in range(np_head):
            # 2. 读取当前 Halo 的粒子数 (4 bytes)
            halo_np_bytes = f.read(4)
            if not halo_np_bytes: break
            halo_np = struct.unpack('i', halo_np_bytes)[0]
            
            if halo_np <= 0: # 约定 0 或负数为结束符
                break
            
            # print(halo_np)
            # 3. 批量读取核心数据
            # ips: 1 * int32 (4 bytes)
            ips = np.fromfile(f, dtype='i8', count=halo_np)
            # xp: 2 * float32
            xp = np.fromfile(f, dtype='f4', count=halo_np * 2).reshape(halo_np, 2)
            # qp: 2 * float32
            qp = np.fromfile(f, dtype='f4', count=halo_np * 2).reshape(halo_np, 2)
            
            # 4. 读取中心点位置
            xp_mean = np.fromfile(f, dtype='f4', count=2)
            qp_mean = np.fromfile(f, dtype='f4', count=2)

            # 5. 长度过滤
            halos.append({
                'np': halo_np,
                'x_center': xp_mean,
                'q_center': qp_mean,
                'xpos': xp,
                'qpos': qp,
                'ip': ips
            })
                
    return halos, b_link, np_halo_min_file, np_head

def fof_fast(xpos, ip, b_link=0.2, np_halo_min=10):
    # 1. 对指定的子集粒子进行聚类
    sub_data = xpos[ip, :]
    db = DBSCAN(eps=b_link*1.01, min_samples=1, metric='euclidean', n_jobs=-1).fit(sub_data)
    
    labels = db.labels_
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    # 2. 筛选符合最小粒子数要求的标签
    mask = counts >= np_halo_min
    if not np.any(mask):
        return None, None  # 如果没有符合条件的 Halo
        
    valid_labels = unique_labels[mask]
    valid_counts = counts[mask]
    
    # 3. 找到粒子数最多的 Halo (最大的 Halo)
    max_idx = np.argmax(valid_counts)
    best_label = valid_labels[max_idx]
    
    # 4. 获取索引映射
    # local_indices 是在 sub_data (即 ip) 中的位置
    is_max_halo = labels == best_label
    
    return is_max_halo

In [ ]:
halos,_,_,_ = read_halo_file(f'{Path}3.000_halo.bin')

idx_halo = [0,1,2]
ips = [halos[i]['ip'] for i in idx_halo]
print(len(ips[0]))

In [ ]:
import numpy as np
frame_ids = list(range(0, 186))
halos,_,_,_ = read_halo_file(f'{Path}3.000_halo.bin')

idx_halo = [0]
ips = [halos[i]['ip'] for i in idx_halo]

pos, n = load2dxpos(f'{Path}/runtime/xpos_0.bin')
n_particles = len(ips[0])

# 初始化：记录每个粒子第一次掉入的步数，-1 表示从未掉入
entry_frames = np.full(n_particles, -1, dtype=int)

# 2. 迭代处理
for i in frame_ids:
    pos, n = load2dxpos(f'{Path}/runtime/xpos_{i:d}.bin')
    
    # 运行 FoF 获取当前 Mask
    # 这里的 current_mask 长度与 target_ips 一致
    current_mask = fof_fast(pos, ips[0])
    
    if current_mask is not None and np.any(current_mask):
        # 判定条件：当前在 Halo 中 (current_mask) 
        # AND 该位置之前从未被记录过 (entry_frames == -1)
        # 这里的 & 符号就是我们要的“只记第一次”的开关
        first_time_mask = current_mask & (entry_frames == -1)
        
        # 只有符合上述两个条件的粒子，才会被填入当前的步数 i
        entry_frames[first_time_mask] = i

# 3. 统计最终进入过 Halo 的粒子 ID 和时间
# 你可以轻松得到一个包含 [粒子ID, 掉入步数] 的对照表
mask_ever_in = entry_frames != -1
captured_ips = ips[0][mask_ever_in]
captured_times = entry_frames[mask_ever_in]

print(f"追踪完成。在 {len(ips[0])} 个候选粒子中，有 {len(captured_ips)} 个粒子最终进入了主集团。")

In [ ]:
ips[j].shape

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl
cmap = plt.get_cmap('turbo')
norm = mpl.colors.Normalize(vmin=min(captured_times), vmax=max(captured_times))
colors = [cmap(norm(entry_step)) for entry_step in captured_times]
mpl.rcParams['animation.embed_limit'] = 1000  # 单位是 MB
fig, ax = plt.subplots(figsize=(4, 4), dpi=150)
pos, n = load2dxpos(f'{Path}/runtime/xpos_0.bin')
halo_pos = pos[captured_ips, :]
sc = ax.scatter([], [], c='b', s=0.01, marker='o', edgecolors='none', alpha=1)
sc_halo = ax.scatter([], [], s=0.1, edgecolors='none')
sc.set_offsets(pos)
sc_halo.set_offsets(halo_pos)
sc_halo.set_facecolors(colors)
ax.axis('equal')
ax.axis('off')
title = ax.set_title('')
ax.set_xlim(-1, n + 1)
ax.set_ylim(-1, n + 1)
ax.set_aspect('equal', adjustable='box')

In [ ]:
# plt xpos s
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl
cmap = plt.get_cmap('turbo')
norm = mpl.colors.Normalize(vmin=min(captured_times), vmax=max(captured_times))
colors = [cmap(norm(entry_step)) for entry_step in captured_times]
mpl.rcParams['animation.embed_limit'] = 1000  # 单位是 MB
fig, ax = plt.subplots(figsize=(4, 4), dpi=150)
pos, n = load2dxpos(f'{Path}/runtime/xpos_0.bin')
sc = ax.scatter([], [], c='b', s=0.01, marker='o', edgecolors='none', alpha=1)
sc_halo = ax.scatter([], [], s=0.1, edgecolors='none')
ax.axis('equal')
ax.axis('off')
title = ax.set_title('')
ax.set_xlim(-1, n + 1)
ax.set_ylim(-1, n + 1)
ax.set_aspect('equal', adjustable='box')

def update(i):
    pos, n = load2dxpos(f'{Path}/runtime/xpos_{i:d}.bin')
    sc.set_offsets(pos)
    halo_pos = pos[captured_ips, :]
    sc_halo.set_offsets(halo_pos)
    sc_halo.set_facecolors(colors)
    title.set_text(f'step {i}')
    return sc,sc_halo, title

ani = FuncAnimation(fig, update, frames=frame_ids[::10], interval=300)

HTML(ani.to_jshtml())  # 在Jupyter中显示动画